# Знакомство с данными Olist

Цель — определить смысл строк, ключи и связи до расчёта бизнес-показателей. Исходные CSV не изменяем.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'scripts'))
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)


## Размеры, пропуски и полные дубликаты

Почтовые префиксы читаем как строки, чтобы сохранить ведущие нули.

In [2]:
summary=[]
profiles=[]
for file in sorted((ROOT/'data').glob('*.csv')):
    cols=pd.read_csv(file,nrows=0).columns
    types={c:'string' for c in cols if 'zip_code_prefix' in c}
    df=pd.read_csv(file,dtype=types)
    summary.append({'file':file.name,'rows':len(df),'columns':len(df.columns),
                    'extra_full_duplicates':int(df.duplicated().sum())})
    for col in df:
        profiles.append({'file':file.name,'column':col,'dtype':str(df[col].dtype),
                         'missing':int(df[col].isna().sum()),
                         'unique_non_null':int(df[col].nunique())})
display(pd.DataFrame(summary))
display(pd.DataFrame(profiles).query('missing > 0'))


,file,rows,columns,extra_full_duplicates
0,olist_customers_dataset.csv,99441,5,0
1,olist_geolocation_dataset.csv,1000163,5,261831
2,olist_order_items_dataset.csv,112650,7,0
3,olist_order_payments_dataset.csv,103886,5,0
4,olist_order_reviews_dataset.csv,99224,7,0
5,olist_orders_dataset.csv,99441,8,0
6,olist_products_dataset.csv,32951,9,0
7,olist_sellers_dataset.csv,3095,4,0
8,product_category_name_translation.csv,71,2,0


,file,column,dtype,missing,unique_non_null
25,olist_order_reviews_dataset.csv,review_comment_title,str,87656,4527
26,olist_order_reviews_dataset.csv,review_comment_message,str,58247,36159
33,olist_orders_dataset.csv,order_approved_at,str,160,90733
34,olist_orders_dataset.csv,order_delivered_carrier_date,str,1783,81018
35,olist_orders_dataset.csv,order_delivered_customer_date,str,2965,95664
38,olist_products_dataset.csv,product_category_name,str,610,73
39,olist_products_dataset.csv,product_name_lenght,float64,610,66
40,olist_products_dataset.csv,product_description_lenght,float64,610,2960
41,olist_products_dataset.csv,product_photos_qty,float64,610,19
42,olist_products_dataset.csv,product_weight_g,float64,2,2204


## Ключи и идентификация покупателя

customer_id связывает запись с заказом. customer_unique_id используется для уникальных покупателей.

In [3]:
customers=pd.read_csv(ROOT/'data/olist_customers_dataset.csv',dtype={'customer_zip_code_prefix':'string'})
print('Записей покупателей:',len(customers))
print('Уникальных customer_id:',customers.customer_id.nunique())
print('Уникальных customer_unique_id:',customers.customer_unique_id.nunique())
display(customers.head())
items=pd.read_csv(ROOT/'data/olist_order_items_dataset.csv')
print('Повторяющихся строк составного ключа order_items:',
      items.duplicated(['order_id','order_item_id'],keep=False).sum())
print('order_id в позициях не уникален:',items.order_id.duplicated().any())


Записей покупателей: 99441
Уникальных customer_id: 99441
Уникальных customer_unique_id: 96096


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


Повторяющихся строк составного ключа order_items: 0
order_id в позициях не уникален: True


## Связи и ограничения

Заказ может иметь несколько позиций, оплат и отзывов. Сначала агрегируем дочерние таблицы, затем JOIN. Геолокация содержит повторяющиеся префиксы и полные дубликаты, поэтому не присоединяем её напрямую. Подробные проверки сохранены в data_understanding.

In [4]:
display(pd.read_csv(ROOT/'reports/data_understanding/relationships.csv'))


,child,child_column,parent,parent_column,observed_cardinality_child_to_parent,child_null_rows,unmatched_non_null_rows,unmatched_distinct_values,parent_non_null_unique
0,orders,customer_id,customers,customer_id,1:1,0,0,0,True
1,order_items,order_id,orders,order_id,many:1,0,0,0,True
2,order_items,product_id,products,product_id,many:1,0,0,0,True
3,order_items,seller_id,sellers,seller_id,many:1,0,0,0,True
4,order_payments,order_id,orders,order_id,many:1,0,0,0,True
5,order_reviews,order_id,orders,order_id,many:1,0,0,0,True
6,products,product_category_name,category_translation,product_category_name,many:1,610,13,2,True
7,customers,customer_zip_code_prefix,geolocation,geolocation_zip_code_prefix,many:many,0,278,157,False
8,sellers,seller_zip_code_prefix,geolocation,geolocation_zip_code_prefix,many:many,0,7,7,False


## Следующий шаг

SQL создаёт raw, загружает CSV, проверяет качество и формирует analytics.order_fact с одной строкой на заказ. Полный словарь: reports/data_understanding/data_dictionary.md.